# AI FinOps - Gestor de Base de Datos SQLite (Desarrollo)

Usa este notebook para consultar y modificar cómodamente las nuevas tablas relacionales (`Rol`, `Usuario`, `Modelo`, `Query`) en la base de datos SQLite (`finops_db.sqlite`).

In [ ]:
import sqlite3
import os

# Ruta al archivo SQLite de la aplicación
db_path = os.path.join('src', 'app', 'finops_db.sqlite')
print(f"Base de datos: {db_path}")
print(f"Existe el archivo: {'Sí' if os.path.exists(db_path) else 'No'}")

### Función auxiliar para consultas
Esta función ejecuta sentencias SQL. Si es un `SELECT`, devuelve un DataFrame de `pandas` (si está instalado) o una lista de diccionarios.

In [ ]:
def run_query(query, params=()):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    try:
        cursor.execute(query, params)
        if query.strip().upper().startswith("SELECT"):
            rows = cursor.fetchall()
            columns = [col[0] for col in cursor.description]
            data = [dict(r) for r in rows]
            try:
                import pandas as pd
                return pd.DataFrame(data, columns=columns)
            except ImportError:
                return data
        else:
            conn.commit()
            print(f"Operación exitosa. Filas afectadas: {cursor.rowcount}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()

### 1. Consultar Presupuestos (Tabla `Rol`)

In [ ]:
run_query("SELECT * FROM Rol")

### 2. Consultar Usuarios y su Gasto (Tabla `Usuario`)

In [ ]:
run_query("SELECT u.Email, u.nombre, u.cuota_utilizada, r.nombre as rol_nombre FROM Usuario u JOIN Rol r ON u.rol = r.Id")

### 3. Consultar Tarifas de Modelos (Tabla `Modelo`)

In [ ]:
run_query("SELECT * FROM Modelo")

### 4. Historial de Transacciones (Tabla `Query`)

In [ ]:
run_query("SELECT * FROM Query ORDER BY Fecha DESC LIMIT 10")

### 5. Modificar Límite de Presupuesto de un Rol
Puedes cambiar el presupuesto asignado a cualquier rol (ej. `marketing`, `producto`) modificando `presupuesto_tokens`.

In [ ]:
# Cambiar límite de presupuesto de Marketing a 15.0 dólares
run_query("UPDATE Rol SET presupuesto_tokens = 15.0 WHERE nombre = 'marketing'")

# Comprobar el cambio
run_query("SELECT * FROM Rol WHERE nombre = 'marketing'")

### 6. Restablecer Base de Datos de Consumo
Si deseas reiniciar el gasto acumulado de todos los usuarios y borrar las consultas:

In [ ]:
# Borrar consultas
run_query("DELETE FROM Query")

# Resetear cuota de usuarios
run_query("UPDATE Usuario SET cuota_utilizada = 0.0")

# Verificar
run_query("SELECT * FROM Usuario")